# Vision Transformer experiments

__`Step 1`__ - Import the required libraries and define the random seed for the code.

In [2]:
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from torchvision import transforms
from PIL import Image

from transformers import ViTImageProcessor, ViTForImageClassification
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

import mlflow

# --- Device setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- Deterministic seed setup ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random seed set to {seed}")

set_seed(42)

Using device: cuda
GPU: NVIDIA L4
Random seed set to 42


__`Step 2`__ - Defining the paths to the data and the project.

In [3]:
# --- Determine project root ---
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

# --- Paths to data ---
wikiart_path = project_root / "data"
train_dir = wikiart_path / "train"
val_dir = wikiart_path / "validation"
test_dir = wikiart_path / "test"

# --- Check paths exist ---
for path in [train_dir, val_dir, test_dir]:
    if not path.exists():
        print(f"Warning: {path} does not exist!")

print(f"Project root: {project_root}")
print(f"WikiArt path: {wikiart_path}")

# --- Dataset parameters ---
BATCH_SIZE = 32
IMG_SIZE = (224, 224)  # Resize images for model input

Project root: /teamspace/studios/this_studio/DeepLearning-NOVAIMS2026
WikiArt path: /teamspace/studios/this_studio/DeepLearning-NOVAIMS2026/data


__`Step 3`__ - Creating Pytorch Datasets and DataLoaders

In [4]:
g = torch.Generator()
g.manual_seed(42)

# --- Load processor ---
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

class ImageFolderDataset(Dataset):
    def __init__(self, path, processor, transform=None):
        self.path = Path(path)
        self.processor = processor
        self.transform = transform
        self.samples = []
        self.classes = sorted([d.name for d in path.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        for cls in self.classes:
            cls_dir = path / cls
            for img_path in cls_dir.glob('*'):
                self.samples.append((img_path, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        encoding = self.processor(images=image, return_tensors="pt")
        pixel_values = encoding['pixel_values'].squeeze()  # remove batch dim
        return pixel_values, label

# --- Define transform ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
])

# --- Create datasets ---
train_dataset = ImageFolderDataset(train_dir, processor, transform)
val_dataset = ImageFolderDataset(val_dir, processor, transform)
test_dataset = ImageFolderDataset(test_dir, processor, transform)

# --- DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

Number of training samples: 9231
Number of validation samples: 1969
Number of test samples: 2000


# Modelling

## Version 1 

__`Step 4`__ - Loading the vision transformer model and creating the experiment in mlflow.

In [5]:
from transformers import ViTConfig

# --- Initialize ViT model for custom number of classes ---
num_labels = len(train_dataset.classes)
config = ViTConfig.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=num_labels
)

model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    config=config,
    ignore_mismatched_sizes=True  # important when num_labels != pretrained
)

model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)
loss_fn = CrossEntropyLoss()
epochs = 10

# Directory to save checkpoints
checkpoint_dir = "checkpoints/v1"
os.makedirs(checkpoint_dir, exist_ok=True)

# Initialize MLflow
mlflow.set_experiment("Vision Transformer v1")
print(f"ViT model initialized with {num_labels} classes")

You passed `num_labels=23` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([23, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([23])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


ViT model initialized with 23 classes


__`Step 5`__ - Training the first version of the model.

In [8]:
best_val_f1 = 0
best_model_path = os.path.join(checkpoint_dir, "best_model.pt")

with mlflow.start_run(run_name="ViT_Training"):
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", 5e-5)

    for epoch in range(epochs):
        start_time = time.time()
        
        # --- Training ---
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []

        for pixel_values, labels in train_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(pixel_values=pixel_values)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        train_loss /= len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')

        # --- Validation ---
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            val_probs = []
            for pixel_values, labels in val_loader:
                pixel_values, labels = pixel_values.to(device), labels.to(device)
                outputs = model(pixel_values=pixel_values)
                loss = loss_fn(outputs.logits, labels)

                val_loss += loss.item()
                val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                val_probs.append(torch.softmax(outputs.logits, dim=1).cpu().numpy())

        val_loss /= len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')

        # Compute AUC (one-vs-rest)
        val_probs = np.vstack(val_probs)
        val_labels_bin = label_binarize(val_labels, classes=range(len(train_dataset.classes)))
        val_auc = roc_auc_score(val_labels_bin, val_probs, multi_class='ovr', average='weighted')

        # --- Save best model ---
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved at epoch {epoch+1} with val_f1: {best_val_f1:.4f}")

        # --- Log metrics to MLflow ---
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_f1", train_f1, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_f1", val_f1, step=epoch)
        mlflow.log_metric("val_auc_ovr", val_auc, step=epoch)

        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - "
              f"loss: {train_loss:.4f} - f1: {train_f1:.4f} - "
              f"val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f} - val_auc: {val_auc:.4f}")

Best model saved at epoch 1 with val_f1: 0.8273
Epoch 1/10 - 221.2s - loss: 0.2835 - f1: 0.9209 - val_loss: 0.5278 - val_f1: 0.8273 - val_auc: 0.9924
Best model saved at epoch 2 with val_f1: 0.8560
Epoch 2/10 - 212.8s - loss: 0.0431 - f1: 0.9946 - val_loss: 0.4414 - val_f1: 0.8560 - val_auc: 0.9942
Best model saved at epoch 3 with val_f1: 0.8635
Epoch 3/10 - 214.6s - loss: 0.0069 - f1: 0.9999 - val_loss: 0.3971 - val_f1: 0.8635 - val_auc: 0.9951
Best model saved at epoch 4 with val_f1: 0.8734
Epoch 4/10 - 214.3s - loss: 0.0027 - f1: 1.0000 - val_loss: 0.3981 - val_f1: 0.8734 - val_auc: 0.9954
Epoch 5/10 - 215.1s - loss: 0.0016 - f1: 1.0000 - val_loss: 0.4058 - val_f1: 0.8708 - val_auc: 0.9953
Epoch 6/10 - 209.5s - loss: 0.0010 - f1: 1.0000 - val_loss: 0.4138 - val_f1: 0.8712 - val_auc: 0.9953
Best model saved at epoch 7 with val_f1: 0.8750
Epoch 7/10 - 212.5s - loss: 0.0007 - f1: 1.0000 - val_loss: 0.4234 - val_f1: 0.8750 - val_auc: 0.9953
Epoch 8/10 - 213.6s - loss: 0.0005 - f1: 1.000

__`Step 6`__ - Checking the results for the test set.

In [9]:
# --- Load best model ---
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_preds = []
test_labels = []
test_probs = []

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)
        probs = torch.softmax(outputs.logits, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())
        test_probs.append(probs.cpu().numpy())

# Metrics
test_labels_bin = label_binarize(test_labels, classes=range(len(train_dataset.classes)))
test_probs = np.vstack(test_probs)

test_f1_macro = f1_score(test_labels, test_preds, average='macro')
test_auc_ovr = roc_auc_score(test_labels_bin, test_probs, multi_class='ovr', average='weighted')

print(f"Test F1-score: {test_f1_macro:.4f}, Test AUC (OvR): {test_auc_ovr:.4f}")

# Log to MLflow
mlflow.log_metric("test_f1_macro", test_f1_macro)
mlflow.log_metric("test_auc_ovr", test_auc_ovr)

Test F1-score: 0.8627, Test AUC (OvR): 0.9942


__`Step 7`__ - Getting the classification report for the training, validation and test set.

In [11]:
from sklearn.metrics import classification_report

print("Metrics for the training set\n")
print(classification_report(
    train_labels,
    train_preds,
    target_names=train_dataset.classes
))

print("Metrics for the validation set\n")
print(classification_report(
    val_labels,
    val_preds,
    target_names=train_dataset.classes
))

print("Metrics for the test set\n")
print(classification_report(
    test_labels,
    test_preds,
    target_names=train_dataset.classes
))

Metrics for the training set

                       precision    recall  f1-score   support

       Albrecht_Durer       1.00      1.00      1.00       396
      Boris_Kustodiev       1.00      1.00      1.00       310
     Camille_Pissarro       1.00      1.00      1.00       427
        Childe_Hassam       1.00      1.00      1.00       266
         Claude_Monet       1.00      1.00      1.00       643
          Edgar_Degas       1.00      1.00      1.00       298
        Eugene_Boudin       1.00      1.00      1.00       272
         Gustave_Dore       1.00      1.00      1.00       366
           Ilya_Repin       1.00      1.00      1.00       264
      Ivan_Aivazovsky       1.00      1.00      1.00       277
        Ivan_Shishkin       1.00      1.00      1.00       254
  John_Singer_Sargent       1.00      1.00      1.00       380
         Marc_Chagall       1.00      1.00      1.00       375
      Martiros_Saryan       1.00      1.00      1.00       282
     Nicholas_Roerich   

## Version 2

__`Step 8`__ - Creating a new version of the model that has droupout in both hidden and attention layers.

In [15]:
from transformers import ViTConfig

# --- Initialize ViT model for custom number of classes ---
num_labels = len(train_dataset.classes)
config = ViTConfig.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=num_labels,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2
)

model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    config=config,
    ignore_mismatched_sizes=True
)

model.to(device)

# --- Updated hyperparameter ---
optimizer = AdamW(model.parameters(), lr=5e-5)  # slightly lower LR for v2
loss_fn = CrossEntropyLoss()
epochs = 10

# --- New checkpoint directory ---
checkpoint_dir = "checkpoints/v2"
os.makedirs(checkpoint_dir, exist_ok=True)

# --- MLflow: new version ---
mlflow.set_experiment("Vision Transformer v2")

print(f"ViT v2 model initialized with {num_labels} classes")

You passed `num_labels=23` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([23, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([23])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


ViT v2 model initialized with 23 classes


__`Step 9`__ - Training the second version of the model.

In [17]:
best_val_f1 = 0
best_model_path = os.path.join(checkpoint_dir, "best_model.pt")

with mlflow.start_run(run_name="ViT_Training2", nested=True):
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", 5e-5)

    for epoch in range(epochs):
        start_time = time.time()
        
        # --- Training ---
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []

        for pixel_values, labels in train_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(pixel_values=pixel_values)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        train_loss /= len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')

        # --- Validation ---
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            val_probs = []
            for pixel_values, labels in val_loader:
                pixel_values, labels = pixel_values.to(device), labels.to(device)
                outputs = model(pixel_values=pixel_values)
                loss = loss_fn(outputs.logits, labels)

                val_loss += loss.item()
                val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                val_probs.append(torch.softmax(outputs.logits, dim=1).cpu().numpy())

        val_loss /= len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')

        # Compute AUC (one-vs-rest)
        val_probs = np.vstack(val_probs)
        val_labels_bin = label_binarize(val_labels, classes=range(len(train_dataset.classes)))
        val_auc = roc_auc_score(val_labels_bin, val_probs, multi_class='ovr', average='weighted')

        # --- Save best model ---
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved at epoch {epoch+1} with val_f1: {best_val_f1:.4f}")

        # --- Log metrics to MLflow ---
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_f1", train_f1, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_f1", val_f1, step=epoch)
        mlflow.log_metric("val_auc_ovr", val_auc, step=epoch)

        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - "
              f"loss: {train_loss:.4f} - f1: {train_f1:.4f} - "
              f"val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f} - val_auc: {val_auc:.4f}")

Best model saved at epoch 1 with val_f1: 0.7272
Epoch 1/10 - 221.5s - loss: 1.3973 - f1: 0.5704 - val_loss: 0.8057 - val_f1: 0.7272 - val_auc: 0.9841
Best model saved at epoch 2 with val_f1: 0.7891
Epoch 2/10 - 213.5s - loss: 0.5829 - f1: 0.8198 - val_loss: 0.6632 - val_f1: 0.7891 - val_auc: 0.9891
Best model saved at epoch 3 with val_f1: 0.8119
Epoch 3/10 - 217.9s - loss: 0.2915 - f1: 0.9167 - val_loss: 0.5689 - val_f1: 0.8119 - val_auc: 0.9920
Best model saved at epoch 4 with val_f1: 0.8125
Epoch 4/10 - 215.3s - loss: 0.1392 - f1: 0.9627 - val_loss: 0.5589 - val_f1: 0.8125 - val_auc: 0.9922
Epoch 5/10 - 215.5s - loss: 0.0945 - f1: 0.9727 - val_loss: 0.6738 - val_f1: 0.7982 - val_auc: 0.9896
Epoch 6/10 - 214.3s - loss: 0.0542 - f1: 0.9862 - val_loss: 0.6915 - val_f1: 0.8062 - val_auc: 0.9898
Epoch 7/10 - 214.5s - loss: 0.0663 - f1: 0.9807 - val_loss: 0.6973 - val_f1: 0.8066 - val_auc: 0.9897
Best model saved at epoch 8 with val_f1: 0.8223
Epoch 8/10 - 215.3s - loss: 0.0458 - f1: 0.986

__`Step 10`__ - Checking the results for the test set.

In [18]:
# --- Load best model ---
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_preds = []
test_labels = []
test_probs = []

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)
        probs = torch.softmax(outputs.logits, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())
        test_probs.append(probs.cpu().numpy())

# Metrics
test_labels_bin = label_binarize(test_labels, classes=range(len(train_dataset.classes)))
test_probs = np.vstack(test_probs)

test_f1_macro = f1_score(test_labels, test_preds, average='macro')
test_auc_ovr = roc_auc_score(test_labels_bin, test_probs, multi_class='ovr', average='weighted')

print(f"Test F1-score: {test_f1_macro:.4f}, Test AUC (OvR): {test_auc_ovr:.4f}")

# Log to MLflow
mlflow.log_metric("test_f1_macro", test_f1_macro)
mlflow.log_metric("test_auc_ovr", test_auc_ovr)

Test F1-score: 0.8265, Test AUC (OvR): 0.9902


__`Step 11`__ - Getting the classification report for the training, validation and test set.

In [19]:
from sklearn.metrics import classification_report

print("Metrics for the training set\n")
print(classification_report(
    train_labels,
    train_preds,
    target_names=train_dataset.classes
))

print("Metrics for the validation set\n")
print(classification_report(
    val_labels,
    val_preds,
    target_names=train_dataset.classes
))

print("Metrics for the test set\n")
print(classification_report(
    test_labels,
    test_preds,
    target_names=train_dataset.classes
))

Metrics for the training set

                       precision    recall  f1-score   support

       Albrecht_Durer       0.99      0.99      0.99       396
      Boris_Kustodiev       0.98      0.96      0.97       310
     Camille_Pissarro       0.97      0.97      0.97       427
        Childe_Hassam       0.98      0.97      0.98       266
         Claude_Monet       0.99      0.99      0.99       643
          Edgar_Degas       0.96      0.96      0.96       298
        Eugene_Boudin       0.98      0.97      0.98       272
         Gustave_Dore       0.99      0.99      0.99       366
           Ilya_Repin       0.98      0.98      0.98       264
      Ivan_Aivazovsky       1.00      1.00      1.00       277
        Ivan_Shishkin       0.98      0.98      0.98       254
  John_Singer_Sargent       0.98      0.99      0.99       380
         Marc_Chagall       0.99      0.98      0.99       375
      Martiros_Saryan       0.98      0.98      0.98       282
     Nicholas_Roerich   

## Version 3

__`Step 8`__ - Creating a new version of the model that has droupout in both hidden and attention layers.

In [5]:
from transformers import ViTConfig

# Initialize ViT model
num_labels = len(train_dataset.classes)
config = ViTConfig.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=num_labels,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2
)

model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    config=config,
    ignore_mismatched_sizes=True
)

model.to(device)

# Optimizer 
optimizer = AdamW(model.parameters(), lr=5e-5)

# Learning rate scheduler
from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=2, gamma=0.5)

loss_fn = CrossEntropyLoss()
epochs = 15

# --- Checkpoints ---
checkpoint_dir = "checkpoints/v3"
os.makedirs(checkpoint_dir, exist_ok=True)

# --- MLflow ---
mlflow.set_experiment("Vision Transformer v3")

print(f"ViT v3 model initialized with LR scheduler")

You passed `num_labels=23` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([23])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([23, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


ViT v3 model initialized with LR scheduler


__`Step 9`__ - Training the second version of the model.

In [6]:
best_val_f1 = 0
best_model_path = os.path.join(checkpoint_dir, "best_model.pt")

with mlflow.start_run(run_name="ViT_Training3", nested=True):
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", 5e-5)

    for epoch in range(epochs):
        start_time = time.time()
        
        # --- Training ---
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []

        for pixel_values, labels in train_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(pixel_values=pixel_values)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        train_loss /= len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')

        # --- Validation ---
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            val_probs = []
            for pixel_values, labels in val_loader:
                pixel_values, labels = pixel_values.to(device), labels.to(device)
                outputs = model(pixel_values=pixel_values)
                loss = loss_fn(outputs.logits, labels)

                val_loss += loss.item()
                val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                val_probs.append(torch.softmax(outputs.logits, dim=1).cpu().numpy())

        val_loss /= len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')

        # Compute AUC (one-vs-rest)
        val_probs = np.vstack(val_probs)
        val_labels_bin = label_binarize(val_labels, classes=range(len(train_dataset.classes)))
        val_auc = roc_auc_score(val_labels_bin, val_probs, multi_class='ovr', average='weighted')

        # --- Save best model ---
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved at epoch {epoch+1} with val_f1: {best_val_f1:.4f}")

        # --- Log metrics to MLflow ---
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_f1", train_f1, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_f1", val_f1, step=epoch)
        mlflow.log_metric("val_auc_ovr", val_auc, step=epoch)

        current_lr = optimizer.param_groups[0]['lr']
        mlflow.log_metric("learning_rate", current_lr, step=epoch)

        scheduler.step()
        
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - "
              f"loss: {train_loss:.4f} - f1: {train_f1:.4f} - "
              f"val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f} - val_auc: {val_auc:.4f}")

Best model saved at epoch 1 with val_f1: 0.7397
Epoch 1/15 - 308.1s - loss: 1.3494 - f1: 0.5821 - val_loss: 0.7983 - val_f1: 0.7397 - val_auc: 0.9835
Best model saved at epoch 2 with val_f1: 0.7819
Epoch 2/15 - 203.1s - loss: 0.5626 - f1: 0.8198 - val_loss: 0.6632 - val_f1: 0.7819 - val_auc: 0.9892
Best model saved at epoch 3 with val_f1: 0.8227
Epoch 3/15 - 204.4s - loss: 0.2214 - f1: 0.9417 - val_loss: 0.5305 - val_f1: 0.8227 - val_auc: 0.9924
Best model saved at epoch 4 with val_f1: 0.8234
Epoch 4/15 - 203.8s - loss: 0.1238 - f1: 0.9717 - val_loss: 0.5256 - val_f1: 0.8234 - val_auc: 0.9926
Best model saved at epoch 5 with val_f1: 0.8327
Epoch 5/15 - 204.1s - loss: 0.0554 - f1: 0.9918 - val_loss: 0.5104 - val_f1: 0.8327 - val_auc: 0.9929
Epoch 6/15 - 203.4s - loss: 0.0353 - f1: 0.9973 - val_loss: 0.5038 - val_f1: 0.8322 - val_auc: 0.9932
Best model saved at epoch 7 with val_f1: 0.8332
Epoch 7/15 - 204.0s - loss: 0.0217 - f1: 0.9993 - val_loss: 0.5171 - val_f1: 0.8332 - val_auc: 0.992

__`Step 10`__ - Checking the results for the test set.

In [7]:
# --- Load best model ---
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_preds = []
test_labels = []
test_probs = []

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)
        probs = torch.softmax(outputs.logits, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())
        test_probs.append(probs.cpu().numpy())

# Metrics
test_labels_bin = label_binarize(test_labels, classes=range(len(train_dataset.classes)))
test_probs = np.vstack(test_probs)

test_f1_macro = f1_score(test_labels, test_preds, average='macro')
test_auc_ovr = roc_auc_score(test_labels_bin, test_probs, multi_class='ovr', average='weighted')

print(f"Test F1-score: {test_f1_macro:.4f}, Test AUC (OvR): {test_auc_ovr:.4f}")

# Log to MLflow
mlflow.log_metric("test_f1_macro", test_f1_macro)
mlflow.log_metric("test_auc_ovr", test_auc_ovr)

Test F1-score: 0.8428, Test AUC (OvR): 0.9925


__`Step 11`__ - Getting the classification report for the training, validation and test set.

In [8]:
from sklearn.metrics import classification_report

print("Metrics for the training set\n")
print(classification_report(
    train_labels,
    train_preds,
    target_names=train_dataset.classes
))

print("Metrics for the validation set\n")
print(classification_report(
    val_labels,
    val_preds,
    target_names=train_dataset.classes
))

print("Metrics for the test set\n")
print(classification_report(
    test_labels,
    test_preds,
    target_names=train_dataset.classes
))

Metrics for the training set

                       precision    recall  f1-score   support

       Albrecht_Durer       1.00      1.00      1.00       396
      Boris_Kustodiev       1.00      1.00      1.00       310
     Camille_Pissarro       1.00      1.00      1.00       427
        Childe_Hassam       1.00      1.00      1.00       266
         Claude_Monet       1.00      1.00      1.00       643
          Edgar_Degas       1.00      1.00      1.00       298
        Eugene_Boudin       1.00      1.00      1.00       272
         Gustave_Dore       1.00      1.00      1.00       366
           Ilya_Repin       1.00      1.00      1.00       264
      Ivan_Aivazovsky       1.00      1.00      1.00       277
        Ivan_Shishkin       1.00      1.00      1.00       254
  John_Singer_Sargent       1.00      1.00      1.00       380
         Marc_Chagall       1.00      1.00      1.00       375
      Martiros_Saryan       1.00      1.00      1.00       282
     Nicholas_Roerich   

## Version 4

__`Step 8`__ - Creating a new version of the model that has droupout in both hidden and attention layers.

In [9]:
from transformers import ViTConfig

# Initialize ViT model
num_labels = len(train_dataset.classes)
config = ViTConfig.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=num_labels,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.2
)

model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    config=config,
    ignore_mismatched_sizes=True
)

model.to(device)

# Optimizer 
optimizer = AdamW(model.parameters(), lr=5e-5)

# Learning rate scheduler
from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=2, gamma=0.5)

loss_fn = CrossEntropyLoss()
epochs = 15

# --- Checkpoints ---
checkpoint_dir = "checkpoints/v4"
os.makedirs(checkpoint_dir, exist_ok=True)

# --- MLflow ---
mlflow.set_experiment("Vision Transformer v4")

print(f"ViT v4 model initialized with LR scheduler")

You passed `num_labels=23` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([23])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([23, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
2026/04/06 16:48:03 INFO mlflow.tracking.fluent: Experiment with name 'Vision Transformer v4' does not exist. Creating a new experiment.


ViT v4 model initialized with LR scheduler


__`Step 9`__ - Training the second version of the model.

In [10]:
best_val_f1 = 0
best_model_path = os.path.join(checkpoint_dir, "best_model.pt")

with mlflow.start_run(run_name="ViT_Training4", nested=True):
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", 5e-5)

    for epoch in range(epochs):
        start_time = time.time()
        
        # --- Training ---
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []

        for pixel_values, labels in train_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(pixel_values=pixel_values)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        train_loss /= len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')

        # --- Validation ---
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            val_probs = []
            for pixel_values, labels in val_loader:
                pixel_values, labels = pixel_values.to(device), labels.to(device)
                outputs = model(pixel_values=pixel_values)
                loss = loss_fn(outputs.logits, labels)

                val_loss += loss.item()
                val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                val_probs.append(torch.softmax(outputs.logits, dim=1).cpu().numpy())

        val_loss /= len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')

        # Compute AUC (one-vs-rest)
        val_probs = np.vstack(val_probs)
        val_labels_bin = label_binarize(val_labels, classes=range(len(train_dataset.classes)))
        val_auc = roc_auc_score(val_labels_bin, val_probs, multi_class='ovr', average='weighted')

        # --- Save best model ---
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved at epoch {epoch+1} with val_f1: {best_val_f1:.4f}")

        # --- Log metrics to MLflow ---
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_f1", train_f1, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_f1", val_f1, step=epoch)
        mlflow.log_metric("val_auc_ovr", val_auc, step=epoch)

        current_lr = optimizer.param_groups[0]['lr']
        mlflow.log_metric("learning_rate", current_lr, step=epoch)

        scheduler.step()
        
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - "
              f"loss: {train_loss:.4f} - f1: {train_f1:.4f} - "
              f"val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f} - val_auc: {val_auc:.4f}")

Best model saved at epoch 1 with val_f1: 0.6648
Epoch 1/15 - 200.8s - loss: 1.4929 - f1: 0.5365 - val_loss: 1.0393 - val_f1: 0.6648 - val_auc: 0.9759
Best model saved at epoch 2 with val_f1: 0.7501
Epoch 2/15 - 203.7s - loss: 0.6970 - f1: 0.7801 - val_loss: 0.7857 - val_f1: 0.7501 - val_auc: 0.9861
Best model saved at epoch 3 with val_f1: 0.7841
Epoch 3/15 - 204.2s - loss: 0.3542 - f1: 0.8938 - val_loss: 0.6770 - val_f1: 0.7841 - val_auc: 0.9889
Best model saved at epoch 4 with val_f1: 0.7878
Epoch 4/15 - 205.8s - loss: 0.2243 - f1: 0.9382 - val_loss: 0.6332 - val_f1: 0.7878 - val_auc: 0.9898
Best model saved at epoch 5 with val_f1: 0.7995
Epoch 5/15 - 203.5s - loss: 0.1231 - f1: 0.9750 - val_loss: 0.6252 - val_f1: 0.7995 - val_auc: 0.9901
Epoch 6/15 - 202.9s - loss: 0.0887 - f1: 0.9826 - val_loss: 0.6589 - val_f1: 0.7858 - val_auc: 0.9901
Epoch 7/15 - 202.0s - loss: 0.0598 - f1: 0.9912 - val_loss: 0.6432 - val_f1: 0.7955 - val_auc: 0.9902
Epoch 8/15 - 203.1s - loss: 0.0490 - f1: 0.994

__`Step 10`__ - Checking the results for the test set.

In [11]:
# --- Load best model ---
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_preds = []
test_labels = []
test_probs = []

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)
        probs = torch.softmax(outputs.logits, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())
        test_probs.append(probs.cpu().numpy())

# Metrics
test_labels_bin = label_binarize(test_labels, classes=range(len(train_dataset.classes)))
test_probs = np.vstack(test_probs)

test_f1_macro = f1_score(test_labels, test_preds, average='macro')
test_auc_ovr = roc_auc_score(test_labels_bin, test_probs, multi_class='ovr', average='weighted')

print(f"Test F1-score: {test_f1_macro:.4f}, Test AUC (OvR): {test_auc_ovr:.4f}")

# Log to MLflow
mlflow.log_metric("test_f1_macro", test_f1_macro)
mlflow.log_metric("test_auc_ovr", test_auc_ovr)

Test F1-score: 0.8054, Test AUC (OvR): 0.9890


__`Step 11`__ - Getting the classification report for the training, validation and test set.

In [12]:
from sklearn.metrics import classification_report

print("Metrics for the training set\n")
print(classification_report(
    train_labels,
    train_preds,
    target_names=train_dataset.classes
))

print("Metrics for the validation set\n")
print(classification_report(
    val_labels,
    val_preds,
    target_names=train_dataset.classes
))

print("Metrics for the test set\n")
print(classification_report(
    test_labels,
    test_preds,
    target_names=train_dataset.classes
))

Metrics for the training set

                       precision    recall  f1-score   support

       Albrecht_Durer       1.00      1.00      1.00       396
      Boris_Kustodiev       1.00      1.00      1.00       310
     Camille_Pissarro       1.00      1.00      1.00       427
        Childe_Hassam       1.00      1.00      1.00       266
         Claude_Monet       1.00      1.00      1.00       643
          Edgar_Degas       1.00      1.00      1.00       298
        Eugene_Boudin       1.00      1.00      1.00       272
         Gustave_Dore       1.00      1.00      1.00       366
           Ilya_Repin       1.00      1.00      1.00       264
      Ivan_Aivazovsky       1.00      1.00      1.00       277
        Ivan_Shishkin       1.00      1.00      1.00       254
  John_Singer_Sargent       1.00      1.00      1.00       380
         Marc_Chagall       1.00      1.00      1.00       375
      Martiros_Saryan       1.00      1.00      1.00       282
     Nicholas_Roerich   

## Version 5

__`Step 8`__ - Creating a new version of the model that has droupout in both hidden and attention layers.

In [16]:
from transformers import ViTConfig

# Initialize ViT model
num_labels = len(train_dataset.classes)
config = ViTConfig.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=num_labels,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2
)

model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    config=config,
    ignore_mismatched_sizes=True
)

model.to(device)

# Optimizer 
optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)

# Learning rate scheduler
from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=2, gamma=0.5)

loss_fn = CrossEntropyLoss(label_smoothing=0.1)
epochs = 15

# --- Checkpoints ---
checkpoint_dir = "checkpoints/v5"
os.makedirs(checkpoint_dir, exist_ok=True)

# --- MLflow ---
mlflow.set_experiment("Vision Transformer v5")

print(f"ViT v5 model initialized with LR scheduler")

You passed `num_labels=23` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([23, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([23])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


2026/04/04 12:42:24 INFO mlflow.tracking.fluent: Experiment with name 'Vision Transformer v5' does not exist. Creating a new experiment.


ViT v5 model initialized with LR scheduler


__`Step 9`__ - Training the second version of the model.

In [17]:
best_val_f1 = 0
best_model_path = os.path.join(checkpoint_dir, "best_model.pt")

with mlflow.start_run(run_name="ViT_Training5", nested=True):
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", 5e-5)

    for epoch in range(epochs):
        start_time = time.time()
        
        # --- Training ---
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []

        for pixel_values, labels in train_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(pixel_values=pixel_values)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        train_loss /= len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')

        # --- Validation ---
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            val_probs = []
            for pixel_values, labels in val_loader:
                pixel_values, labels = pixel_values.to(device), labels.to(device)
                outputs = model(pixel_values=pixel_values)
                loss = loss_fn(outputs.logits, labels)

                val_loss += loss.item()
                val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                val_probs.append(torch.softmax(outputs.logits, dim=1).cpu().numpy())

        val_loss /= len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')

        # Compute AUC (one-vs-rest)
        val_probs = np.vstack(val_probs)
        val_labels_bin = label_binarize(val_labels, classes=range(len(train_dataset.classes)))
        val_auc = roc_auc_score(val_labels_bin, val_probs, multi_class='ovr', average='weighted')

        # --- Save best model ---
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved at epoch {epoch+1} with val_f1: {best_val_f1:.4f}")

        # --- Log metrics to MLflow ---
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_f1", train_f1, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_f1", val_f1, step=epoch)
        mlflow.log_metric("val_auc_ovr", val_auc, step=epoch)

        current_lr = optimizer.param_groups[0]['lr']
        mlflow.log_metric("learning_rate", current_lr, step=epoch)

        scheduler.step()
        
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - "
              f"loss: {train_loss:.4f} - f1: {train_f1:.4f} - "
              f"val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f} - val_auc: {val_auc:.4f}")

Best model saved at epoch 1 with val_f1: 0.7320
Epoch 1/15 - 201.9s - loss: 1.7029 - f1: 0.5859 - val_loss: 1.2843 - val_f1: 0.7320 - val_auc: 0.9831
Best model saved at epoch 2 with val_f1: 0.7906
Epoch 2/15 - 203.1s - loss: 1.0805 - f1: 0.8339 - val_loss: 1.1631 - val_f1: 0.7906 - val_auc: 0.9874
Best model saved at epoch 3 with val_f1: 0.8235
Epoch 3/15 - 203.2s - loss: 0.8271 - f1: 0.9431 - val_loss: 1.0898 - val_f1: 0.8235 - val_auc: 0.9898
Epoch 4/15 - 202.0s - loss: 0.7464 - f1: 0.9747 - val_loss: 1.1173 - val_f1: 0.8041 - val_auc: 0.9891
Epoch 5/15 - 201.4s - loss: 0.6894 - f1: 0.9934 - val_loss: 1.0921 - val_f1: 0.8165 - val_auc: 0.9898
Epoch 6/15 - 201.1s - loss: 0.6716 - f1: 0.9971 - val_loss: 1.0784 - val_f1: 0.8180 - val_auc: 0.9896
Best model saved at epoch 7 with val_f1: 0.8331
Epoch 7/15 - 202.7s - loss: 0.6561 - f1: 0.9987 - val_loss: 1.0439 - val_f1: 0.8331 - val_auc: 0.9904
Epoch 8/15 - 201.4s - loss: 0.6513 - f1: 0.9989 - val_loss: 1.0529 - val_f1: 0.8221 - val_auc:

__`Step 10`__ - Checking the results for the test set.

In [18]:
# --- Load best model ---
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_preds = []
test_labels = []
test_probs = []

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)
        probs = torch.softmax(outputs.logits, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())
        test_probs.append(probs.cpu().numpy())

# Metrics
test_labels_bin = label_binarize(test_labels, classes=range(len(train_dataset.classes)))
test_probs = np.vstack(test_probs)

test_f1_macro = f1_score(test_labels, test_preds, average='macro')
test_auc_ovr = roc_auc_score(test_labels_bin, test_probs, multi_class='ovr', average='weighted')

print(f"Test F1-score: {test_f1_macro:.4f}, Test AUC (OvR): {test_auc_ovr:.4f}")

# Log to MLflow
mlflow.log_metric("test_f1_macro", test_f1_macro)
mlflow.log_metric("test_auc_ovr", test_auc_ovr)

Test F1-score: 0.8373, Test AUC (OvR): 0.9909


__`Step 11`__ - Getting the classification report for the training, validation and test set.

In [19]:
from sklearn.metrics import classification_report

print("Metrics for the training set\n")
print(classification_report(
    train_labels,
    train_preds,
    target_names=train_dataset.classes
))

print("Metrics for the validation set\n")
print(classification_report(
    val_labels,
    val_preds,
    target_names=train_dataset.classes
))

print("Metrics for the test set\n")
print(classification_report(
    test_labels,
    test_preds,
    target_names=train_dataset.classes
))

Metrics for the training set

                       precision    recall  f1-score   support

       Albrecht_Durer       1.00      1.00      1.00       396
      Boris_Kustodiev       1.00      1.00      1.00       310
     Camille_Pissarro       1.00      1.00      1.00       427
        Childe_Hassam       1.00      1.00      1.00       266
         Claude_Monet       1.00      1.00      1.00       643
          Edgar_Degas       1.00      1.00      1.00       298
        Eugene_Boudin       1.00      1.00      1.00       272
         Gustave_Dore       1.00      1.00      1.00       366
           Ilya_Repin       1.00      1.00      1.00       264
      Ivan_Aivazovsky       1.00      1.00      1.00       277
        Ivan_Shishkin       1.00      1.00      1.00       254
  John_Singer_Sargent       1.00      1.00      1.00       380
         Marc_Chagall       1.00      1.00      1.00       375
      Martiros_Saryan       1.00      1.00      1.00       282
     Nicholas_Roerich   

## Version 6

__`Step 8`__ - Creating a new version of the model that has droupout in both hidden and attention layers.

In [13]:
from transformers import ViTConfig

# Initialize ViT model
num_labels = len(train_dataset.classes)
config = ViTConfig.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=num_labels,
    hidden_dropout_prob=0.4,
    attention_probs_dropout_prob=0.4
)

model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    config=config,
    ignore_mismatched_sizes=True
)

model.to(device)

# Optimizer 
optimizer = AdamW(model.parameters(), lr=5e-5)

# Learning rate scheduler
from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=3, gamma=0.5)

loss_fn = CrossEntropyLoss()
epochs = 20

# --- Checkpoints ---
checkpoint_dir = "checkpoints/v6"
os.makedirs(checkpoint_dir, exist_ok=True)

# --- MLflow ---
mlflow.set_experiment("Vision Transformer v6")

print(f"ViT v6 model initialized with LR scheduler")

You passed `num_labels=23` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([23])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([23, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
2026/04/06 17:40:53 INFO mlflow.tracking.fluent: Experiment with name 'Vision Transformer v6' does not exist. Creating a new experiment.


ViT v6 model initialized with LR scheduler


__`Step 9`__ - Training the second version of the model.

In [14]:
best_val_f1 = 0
best_model_path = os.path.join(checkpoint_dir, "best_model.pt")

with mlflow.start_run(run_name="ViT_Training6", nested=True):
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", 5e-5)

    for epoch in range(epochs):
        start_time = time.time()
        
        # --- Training ---
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []

        for pixel_values, labels in train_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(pixel_values=pixel_values)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        train_loss /= len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')

        # --- Validation ---
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            val_probs = []
            for pixel_values, labels in val_loader:
                pixel_values, labels = pixel_values.to(device), labels.to(device)
                outputs = model(pixel_values=pixel_values)
                loss = loss_fn(outputs.logits, labels)

                val_loss += loss.item()
                val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                val_probs.append(torch.softmax(outputs.logits, dim=1).cpu().numpy())

        val_loss /= len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')

        # Compute AUC (one-vs-rest)
        val_probs = np.vstack(val_probs)
        val_labels_bin = label_binarize(val_labels, classes=range(len(train_dataset.classes)))
        val_auc = roc_auc_score(val_labels_bin, val_probs, multi_class='ovr', average='weighted')

        # --- Save best model ---
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved at epoch {epoch+1} with val_f1: {best_val_f1:.4f}")

        # --- Log metrics to MLflow ---
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_f1", train_f1, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_f1", val_f1, step=epoch)
        mlflow.log_metric("val_auc_ovr", val_auc, step=epoch)

        current_lr = optimizer.param_groups[0]['lr']
        mlflow.log_metric("learning_rate", current_lr, step=epoch)

        scheduler.step()
        
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - "
              f"loss: {train_loss:.4f} - f1: {train_f1:.4f} - "
              f"val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f} - val_auc: {val_auc:.4f}")

Best model saved at epoch 1 with val_f1: 0.4352
Epoch 1/20 - 201.9s - loss: 1.8622 - f1: 0.4169 - val_loss: 1.6572 - val_f1: 0.4352 - val_auc: 0.9422
Best model saved at epoch 2 with val_f1: 0.5836
Epoch 2/20 - 203.9s - loss: 1.0032 - f1: 0.6773 - val_loss: 1.2441 - val_f1: 0.5836 - val_auc: 0.9653
Best model saved at epoch 3 with val_f1: 0.6543
Epoch 3/20 - 204.9s - loss: 0.7255 - f1: 0.7709 - val_loss: 1.0249 - val_f1: 0.6543 - val_auc: 0.9752
Best model saved at epoch 4 with val_f1: 0.6908
Epoch 4/20 - 204.2s - loss: 0.4686 - f1: 0.8538 - val_loss: 0.8989 - val_f1: 0.6908 - val_auc: 0.9804
Best model saved at epoch 5 with val_f1: 0.7295
Epoch 5/20 - 203.7s - loss: 0.3730 - f1: 0.8840 - val_loss: 0.8493 - val_f1: 0.7295 - val_auc: 0.9819
Best model saved at epoch 6 with val_f1: 0.7324
Epoch 6/20 - 203.2s - loss: 0.2962 - f1: 0.9104 - val_loss: 0.8088 - val_f1: 0.7324 - val_auc: 0.9834
Best model saved at epoch 7 with val_f1: 0.7351
Epoch 7/20 - 203.3s - loss: 0.1938 - f1: 0.9472 - va

__`Step 10`__ - Checking the results for the test set.

In [15]:
# --- Load best model ---
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_preds = []
test_labels = []
test_probs = []

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)
        probs = torch.softmax(outputs.logits, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())
        test_probs.append(probs.cpu().numpy())

# Metrics
test_labels_bin = label_binarize(test_labels, classes=range(len(train_dataset.classes)))
test_probs = np.vstack(test_probs)

test_f1_macro = f1_score(test_labels, test_preds, average='macro')
test_auc_ovr = roc_auc_score(test_labels_bin, test_probs, multi_class='ovr', average='weighted')

print(f"Test F1-score: {test_f1_macro:.4f}, Test AUC (OvR): {test_auc_ovr:.4f}")

# Log to MLflow
mlflow.log_metric("test_f1_macro", test_f1_macro)
mlflow.log_metric("test_auc_ovr", test_auc_ovr)

Test F1-score: 0.7495, Test AUC (OvR): 0.9837


__`Step 11`__ - Getting the classification report for the training, validation and test set.

In [16]:
from sklearn.metrics import classification_report

print("Metrics for the training set\n")
print(classification_report(
    train_labels,
    train_preds,
    target_names=train_dataset.classes
))

print("Metrics for the validation set\n")
print(classification_report(
    val_labels,
    val_preds,
    target_names=train_dataset.classes
))

print("Metrics for the test set\n")
print(classification_report(
    test_labels,
    test_preds,
    target_names=train_dataset.classes
))

Metrics for the training set

                       precision    recall  f1-score   support

       Albrecht_Durer       0.99      0.99      0.99       396
      Boris_Kustodiev       0.98      1.00      0.99       310
     Camille_Pissarro       1.00      0.99      1.00       427
        Childe_Hassam       1.00      1.00      1.00       266
         Claude_Monet       1.00      1.00      1.00       643
          Edgar_Degas       0.99      0.98      0.99       298
        Eugene_Boudin       1.00      1.00      1.00       272
         Gustave_Dore       1.00      1.00      1.00       366
           Ilya_Repin       0.99      0.98      0.98       264
      Ivan_Aivazovsky       1.00      1.00      1.00       277
        Ivan_Shishkin       1.00      0.99      1.00       254
  John_Singer_Sargent       0.99      0.99      0.99       380
         Marc_Chagall       1.00      1.00      1.00       375
      Martiros_Saryan       1.00      0.99      0.99       282
     Nicholas_Roerich   

# Model 7

In [17]:
import os
import random
import numpy as np
import torch
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from transformers import ViTConfig, ViTForImageClassification, get_cosine_schedule_with_warmup
import mlflow

# -----------------------
# Reproducibility
# -----------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# -----------------------
# Device
# -----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# ViT Model
# -----------------------
num_labels = len(train_dataset.classes)
config = ViTConfig.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=num_labels,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2,
    transformer=dict(drop_path_rate=0.1)  # stochastic depth
)

model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    config=config,
    ignore_mismatched_sizes=True
)

model.to(device)

# -----------------------
# Optimizer & Scheduler
# -----------------------
optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)
epochs = 20
num_training_steps = len(train_loader) * epochs
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps
)

# -----------------------
# Loss
# -----------------------
loss_fn = CrossEntropyLoss(label_smoothing=0.1)

# -----------------------
# Gradual unfreeze
# -----------------------
for name, param in model.vit.named_parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

# -----------------------
# Checkpoints & MLflow
# -----------------------
checkpoint_dir = "checkpoints/v7"
os.makedirs(checkpoint_dir, exist_ok=True)
mlflow.set_experiment("Vision Transformer v7")

print("ViT v7 model initialized with dropout 0.2, stochastic depth, and cosine LR scheduler")

You passed `num_labels=23` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([23])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([23, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
2026/04/06 18:48:54 INFO mlflow.tracking.fluent: Experiment with name 'Vision Transformer v7' does not exist. Creating a new experiment.


ViT v7 model initialized with dropout 0.4, stochastic depth, and cosine LR scheduler


In [18]:
from timm.data.mixup import Mixup
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import time
import numpy as np

# --- Mixup / CutMix ---
mixup_fn = Mixup(
    mixup_alpha=0.8, cutmix_alpha=1.0, prob=1.0, switch_prob=0.5,
    label_smoothing=0.1, num_classes=len(train_dataset.classes)
)

best_val_f1 = 0
best_model_path = os.path.join(checkpoint_dir, "best_model.pt")

with mlflow.start_run(run_name="ViT_Training7", nested=True):
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", 5e-5)

    for epoch in range(epochs):
        start_time = time.time()
        
        # --- Training ---
        model.train()
        train_loss = 0
        train_preds, train_labels = [], []

        for pixel_values, labels in train_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            
            # Mixup
            pixel_values, labels = mixup_fn(pixel_values, labels)
            
            optimizer.zero_grad()
            outputs = model(pixel_values=pixel_values)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            train_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        train_loss /= len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')

        # --- Validation ---
        model.eval()
        val_loss = 0
        val_preds, val_labels, val_probs = [], [], []

        with torch.no_grad():
            for pixel_values, labels in val_loader:
                pixel_values, labels = pixel_values.to(device), labels.to(device)
                outputs = model(pixel_values=pixel_values)
                loss = loss_fn(outputs.logits, labels)

                val_loss += loss.item()
                val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                val_probs.append(torch.softmax(outputs.logits, dim=1).cpu().numpy())

        val_loss /= len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')

        # Compute weighted OvR AUC
        val_probs = np.vstack(val_probs)
        val_labels_bin = label_binarize(val_labels, classes=range(len(train_dataset.classes)))
        val_auc = roc_auc_score(val_labels_bin, val_probs, multi_class='ovr', average='weighted')

        # Save best model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved at epoch {epoch+1} with val_f1: {best_val_f1:.4f}")

        # Log metrics to MLflow
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_f1", train_f1, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_f1", val_f1, step=epoch)
        mlflow.log_metric("val_auc_ovr", val_auc, step=epoch)
        mlflow.log_metric("learning_rate", optimizer.param_groups[0]['lr'], step=epoch)

        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - "
              f"loss: {train_loss:.4f} - f1: {train_f1:.4f} - "
              f"val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f} - val_auc: {val_auc:.4f}")

AssertionError: Batch size should be even when using this

In [ ]:
# --- Load best model ---
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_preds, test_labels, test_probs = [], [], []

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)
        probs = torch.softmax(outputs.logits, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())
        test_probs.append(probs.cpu().numpy())

# Metrics
test_labels_bin = label_binarize(test_labels, classes=range(len(train_dataset.classes)))
test_probs = np.vstack(test_probs)

test_f1_macro = f1_score(test_labels, test_preds, average='macro')
test_auc_ovr = roc_auc_score(test_labels_bin, test_probs, multi_class='ovr', average='weighted')

print(f"Test F1-score: {test_f1_macro:.4f}, Test AUC (OvR): {test_auc_ovr:.4f}")

# Log to MLflow
mlflow.log_metric("test_f1_macro", test_f1_macro)
mlflow.log_metric("test_auc_ovr", test_auc_ovr)

In [ ]:
from sklearn.metrics import classification_report

print("Metrics for the training set\n")
print(classification_report(
    train_labels,
    train_preds,
    target_names=train_dataset.classes
))

print("Metrics for the validation set\n")
print(classification_report(
    val_labels,
    val_preds,
    target_names=train_dataset.classes
))

print("Metrics for the test set\n")
print(classification_report(
    test_labels,
    test_preds,
    target_names=train_dataset.classes
))